In [1]:
import torch
from torch_geometric.data import Data

In [2]:
from pathlib import Path
import numpy as np


from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdmolops import GetAdjacencyMatrix
from pubchemfp import GetPubChemFPs

### 1.Dataset

In [3]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/home/adem/Downloads/Drug Discovery journey/Mastering Generative_ml models")

# Find all CSV files recursively under BASE_DIR
csv_files = sorted(BASE_DIR.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found under {BASE_DIR}")

# List them with indices
for i, path in enumerate(csv_files):
    rel_path = path.relative_to(BASE_DIR)
    size_kb = path.stat().st_size / 1024
    print(f"[{i}] {rel_path}  ({size_kb:.1f} KB)")

# Ask the user which one to load
while True:
    try:
        selected_index = int(input("Enter the index of the CSV to load: "))
        if 0 <= selected_index < len(csv_files):
            break
        else:
            print(f"Please enter a number between 0 and {len(csv_files) - 1}.")
    except ValueError:
        print("Please enter a valid integer.")

csv_path = csv_files[selected_index]
print(f"Loading: {csv_path}")

cdk2_compounds_df = pd.read_csv(csv_path)
cdk2_compounds_df.head()

[0] DGCL-main/Attention/data/pretrain/data/.ipynb_checkpoints/now-checkpoint.csv  (22016.0 KB)
[1] DGCL-main/Attention/data/pretrain/data/now.csv  (22016.0 KB)
[2] DGCL-main/Attention/dataset/bace/raw/smiles.csv  (102.3 KB)
[3] DGCL-main/Attention/dataset/bbbp/raw/smiles.csv  (111.1 KB)
[4] DGCL-main/Attention/dataset/clintox/processed/smiles.csv  (85.0 KB)
[5] DGCL-main/Attention/dataset/clintox/raw/smiles.csv  (94.7 KB)
[6] DGCL-main/Attention/dataset/ecoli/raw/smiles.csv  (106.7 KB)
[7] DGCL-main/Attention/dataset/esol/raw/smiles.csv  (33.6 KB)
[8] DGCL-main/Attention/dataset/freesolv/raw/smiles.csv  (14.5 KB)
[9] DGCL-main/Attention/dataset/hiv/raw/smiles.csv  (2436.5 KB)
[10] DGCL-main/Attention/dataset/lipo/raw/smiles.csv  (222.2 KB)
[11] DGCL-main/Attention/dataset/qm7/raw/smiles.csv  (406.4 KB)
[12] DGCL-main/Attention/dataset/sider/raw/smiles.csv  (185.8 KB)
[13] DGCL-main/Attention/dataset/tox21/raw/smiles.csv  (455.4 KB)
[14] DGCL-main/Attention/dataset/toxcast/raw/smiles.cs

Loading: /home/adem/Downloads/Drug Discovery journey/Mastering Generative_ml models/Phase_0/VAE/project_01/phase_01/data/CDK_compounds_lipinski_NOPAINS_NOBRENKS.csv


,molecule_chembl_id,kinase,IC50,units,canonical_smiles,pIC50,ro5_fulfilled,ROMol,has_brenk
0,CHEMBL3694408,CDK9,0.005,nM,COc1ccc(F)cc1-c1cc(NC(=O)COc2ccc(Cl)cn2)ncn1,11.301030,True,<rdkit.Chem.rdchem.Mol object at 0x735c78f67f40>,False
1,CHEMBL334285,CDK4,0.007,nM,O=C(Nc1cccc2c1C(=O)c1c(-c3ccc(C(=O)N4CCNCC4)s3...,11.154902,True,<rdkit.Chem.rdchem.Mol object at 0x735c78f67b50>,False
2,CHEMBL3694406,CDK9,0.015,nM,COc1ccc(F)cc1-c1cc(NC(=O)Cc2cccnc2)ncn1,10.823909,True,<rdkit.Chem.rdchem.Mol object at 0x735c78f67530>,False
3,CHEMBL3694400,CDK9,0.018,nM,COc1ccc(F)cc1-c1cc(NC(=O)C2CCCNC2)ncn1,10.744727,True,<rdkit.Chem.rdchem.Mol object at 0x735c78f642e0>,False
4,CHEMBL3694402,CDK9,0.019,nM,COc1ccccc1-c1cc(NC(=O)C2CCC(=O)NC2)ncn1,10.721246,True,<rdkit.Chem.rdchem.Mol object at 0x735c78f67e60>,False


### 2.Featurisation 

featurization/encoding consist of converting each molecule's SMILES (already parsed into RDKit Mol objects in your cdk2_compounds_df) into **PyTorch Geometric Data objects** that your GAT/GIN encoders can consume.

Data is PyTorch Geometric's container class for a single graph, and it's flexible by design: you decide what goes into it by choosing which node-level and edge-level information to compute and attach.

#### Molecule → Graph Transform

Starting from **5074 molecules**, each row already has a parsed **`ROMol`** object and a precomputed **`pIC50`** target. This step converts every molecule into a PyTorch Geometric graph object by building four tensors:

- **`x`** - the **atom feature matrix**, one row per atom, encoding properties like **symbol**, **degree**, **formal charge**, and **aromaticity**
- **`edge_index`** - the **bond connectivity**, stored as a **sparse adjacency list** with **both directions** represented (undirected graph)
- **`edge_attr`** - the **bond feature matrix**, encoding **bond type**, **conjugation**, and **ring membership**
- **`y`** - the **`pIC50`** value, attached as the **regression label** for that molecule

Each molecule becomes a single `torch_geometric.data.Data` object bundling `x`, `edge_index`, `edge_attr`, and `y` - ready to be batched by a `DataLoader` for training the **GAT**/**GIN** predictor.

#### Step 1: Atom Featurisation

We start by defining an auxiliary function which transforms a value x into a one-hot encoding based on a list of permitted values for x:

In [4]:
def one_hot_encoding(x, permitted_list):
    """
    Maps input elements x which are not in the permitted list to the last element
    of the permitted list.
    """
    if x not in permitted_list:
        x = permitted_list[-1]
    binary_encoding = [int(boolean_value) for boolean_value in list(map(lambda s: x == s, permitted_list))]
    return binary_encoding

Now we use this auxiliary function to define the actual atom featurisation function:



In [5]:
def get_atom_features(atom, 
                      use_chirality = True, 
                      hydrogens_implicit = True):
    """
    Takes an RDKit atom object as input and gives a 1d-numpy array of atom features as output.
    """
    # define list of permitted atoms
    
    permitted_list_of_atoms =  ['C','N','O','S','F','Si','P','Cl','Br','Mg','Na','Ca','Fe','As','Al','I', 'B','V','K','Tl','Yb','Sb','Sn','Ag','Pd','Co','Se','Ti','Zn', 'Li','Ge','Cu','Au','Ni','Cd','In','Mn','Zr','Cr','Pt','Hg','Pb','Unknown']
    
    if hydrogens_implicit == False:
        permitted_list_of_atoms = ['H'] + permitted_list_of_atoms
    
    # compute atom features
    
    atom_type_enc = one_hot_encoding(str(atom.GetSymbol()), permitted_list_of_atoms)
    
    n_heavy_neighbors_enc = one_hot_encoding(int(atom.GetDegree()), [0, 1, 2, 3, 4, "MoreThanFour"])
    
    formal_charge_enc = one_hot_encoding(int(atom.GetFormalCharge()), [-3, -2, -1, 0, 1, 2, 3, "Extreme"])
    
    hybridisation_type_enc = one_hot_encoding(str(atom.GetHybridization()), ["S", "SP", "SP2", "SP3", "SP3D", "SP3D2", "OTHER"])
    
    is_in_a_ring_enc = [int(atom.IsInRing())]
    
    is_aromatic_enc = [int(atom.GetIsAromatic())]
    
    atomic_mass_scaled = [float((atom.GetMass() - 10.812)/116.092)]
    
    vdw_radius_scaled = [float((Chem.GetPeriodicTable().GetRvdw(atom.GetAtomicNum()) - 1.5)/0.6)]
    
    covalent_radius_scaled = [float((Chem.GetPeriodicTable().GetRcovalent(atom.GetAtomicNum()) - 0.64)/0.76)]
    atom_feature_vector = atom_type_enc + n_heavy_neighbors_enc + formal_charge_enc + hybridisation_type_enc + is_in_a_ring_enc + is_aromatic_enc + atomic_mass_scaled + vdw_radius_scaled + covalent_radius_scaled
                                    
    if use_chirality == True:
        chirality_type_enc = one_hot_encoding(str(atom.GetChiralTag()), ["CHI_UNSPECIFIED", "CHI_TETRAHEDRAL_CW", "CHI_TETRAHEDRAL_CCW", "CHI_OTHER"])
        atom_feature_vector += chirality_type_enc
    
    if hydrogens_implicit == True:
        n_hydrogens_enc = one_hot_encoding(int(atom.GetTotalNumHs()), [0, 1, 2, 3, 4, "MoreThanFour"])
        atom_feature_vector += n_hydrogens_enc
    return np.array(atom_feature_vector)

#### Atom Feature Vector Explanation
The `get_atom_features` function converts a single RDKit atom into a **numerical feature vector** per atom, used as the representation of a node in the **molecular graph**. Categorical properties are **one-hot encoded**, continuous properties are **scaled**, so all features share a similar range - important for stable GNN training.
##### 1. Atom type - `atom_type_enc`
```python
permitted_list_of_atoms = ['C','N','O','S','F','Si','P','Cl','Br', ..., 'Unknown']
atom_type_enc = one_hot_encoding(str(atom.GetSymbol()), permitted_list_of_atoms)
```
One-hot vector over ~43 element symbols. This is the **most important atom-level feature** - it encodes *what element this atom is*. The `'Unknown'` bucket catches elements not in the list.
##### 2. Heavy atom degree - `n_heavy_neighbors_enc`
```python
n_heavy_neighbors_enc = one_hot_encoding(int(atom.GetDegree()), [0, 1, 2, 3, 4, "MoreThanFour"])
```
`GetDegree()` counts **heavy-atom (non-hydrogen) neighbors**, capturing local connectivity.
##### 3. Formal charge - `formal_charge_enc`
```python
formal_charge_enc = one_hot_encoding(int(atom.GetFormalCharge()), [-3, -2, -1, 0, 1, 2, 3, "Extreme"])
```
Captures **charged species** (e.g. carboxylate O⁻, ammonium N⁺) - chemically important for solubility and binding.
##### 4. Hybridisation - `hybridisation_type_enc`
```python
hybridisation_type_enc = one_hot_encoding(str(atom.GetHybridization()), ["S", "SP", "SP2", "SP3", "SP3D", "SP3D2", "OTHER"])
```
Encodes **orbital geometry** (sp, sp2, sp3, etc.), directly related to bond angles and molecular shape.
##### 5. Ring membership - `is_in_a_ring_enc`
```python
is_in_a_ring_enc = [int(atom.IsInRing())]
```
Binary flag: is this atom part of a **ring system** (aromatic or aliphatic)?
##### 6. Aromaticity - `is_aromatic_enc`
```python
is_aromatic_enc = [int(atom.GetIsAromatic())]
```
Binary flag for **aromatic character** (e.g. benzene-ring carbons), tied to electronic and reactivity properties.
##### 7–9. Scaled numerical features
```python
atomic_mass_scaled     = [float((atom.GetMass() - 10.812)/116.092)]
vdw_radius_scaled      = [float((Chem.GetPeriodicTable().GetRvdw(atom.GetAtomicNum()) - 1.5)/0.6)]
covalent_radius_scaled = [float((Chem.GetPeriodicTable().GetRcovalent(atom.GetAtomicNum()) - 0.64)/0.76)]
```
Each is **min-max normalized** using fixed empirical constants, so **atomic mass**, **Van der Waals radius**, and **covalent radius** land in a similar numeric range instead of dominating the vector with large raw values.
##### Concatenation
```python
atom_feature_vector = (atom_type_enc + n_heavy_neighbors_enc + formal_charge_enc + hybridisation_type_enc + is_in_a_ring_enc + is_aromatic_enc + atomic_mass_scaled + vdw_radius_scaled + covalent_radius_scaled)
```
All feature blocks are lists, so `+` performs **list concatenation**, building one flat vector.
##### Optional: chirality
```python
if use_chirality == True:
    chirality_type_enc = one_hot_encoding(str(atom.GetChiralTag()), ["CHI_UNSPECIFIED", "CHI_TETRAHEDRAL_CW", "CHI_TETRAHEDRAL_CCW", "CHI_OTHER"])
    atom_feature_vector += chirality_type_enc
```
Encodes **stereochemistry** (R/S configuration) - important since many drugs have chirality-dependent activity, but optional since it adds complexity.
##### Optional: explicit hydrogens
```python
if hydrogens_implicit == True:
    n_hydrogens_enc = one_hot_encoding(int(atom.GetTotalNumHs()), [0, 1, 2, 3, 4, "MoreThanFour"])
    atom_feature_vector += n_hydrogens_enc
```
If hydrogens are **implicit** (not their own graph nodes), this feature encodes **how many hydrogens are attached** instead.
##### Return
```python
return np.array(atom_feature_vector)
```
The flat list is cast to a **NumPy array** - a convenient per-atom (or per-bond) format that gets stacked across all atoms/bonds and converted to a `torch.Tensor` when building the graph's `Data` object.

#### Step 2: Bond Featurisation

In [6]:
def get_bond_features(bond, 
                      use_stereochemistry = True):
    """
    Takes an RDKit bond object as input and gives a 1d-numpy array of bond features as output.
    """

    permitted_list_of_bond_types = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]

    bond_type_enc = one_hot_encoding(bond.GetBondType(), permitted_list_of_bond_types)
    
    bond_is_conj_enc = [int(bond.GetIsConjugated())]
    
    bond_is_in_ring_enc = [int(bond.IsInRing())]
    
    bond_feature_vector = bond_type_enc + bond_is_conj_enc + bond_is_in_ring_enc
    
    if use_stereochemistry == True:
        stereo_type_enc = one_hot_encoding(str(bond.GetStereo()), ["STEREOZ", "STEREOE", "STEREOANY", "STEREONONE"])
        bond_feature_vector += stereo_type_enc

    return np.array(bond_feature_vector)

#### Bond Feature Vector Explanation
The `get_bond_features` function converts a single RDKit bond into a **numerical feature vector**, used as the representation of an edge in the **molecular graph**. Same pattern as atom features: categorical properties are **one-hot encoded**.
##### 1. Bond type - `bond_type_enc`
```python
permitted_list_of_bond_types = [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
bond_type_enc = one_hot_encoding(bond.GetBondType(), permitted_list_of_bond_types)
```
One-hot vector over the four possible **bond orders**: single, double, triple, aromatic. This is the **core bond feature** - it tells the model what kind of connection links the two atoms.
##### 2. Conjugation - `bond_is_conj_enc`
```python
bond_is_conj_enc = [int(bond.GetIsConjugated())]
```
Binary flag: is this bond part of a **conjugated system** (alternating single/double bonds allowing electron delocalization)? Relevant to reactivity and electronic properties.
##### 3. Ring membership - `bond_is_in_ring_enc`
```python
bond_is_in_ring_enc = [int(bond.IsInRing())]
```
Binary flag: is this bond part of a **ring**? Complements the atom-level `is_in_a_ring_enc` feature.
##### Concatenation
```python
bond_feature_vector = bond_type_enc + bond_is_conj_enc + bond_is_in_ring_enc
```
List concatenation combines the three feature blocks into one flat vector.
##### Optional: stereochemistry
```python
if use_stereochemistry == True:
    stereo_type_enc = one_hot_encoding(str(bond.GetStereo()), ["STEREOZ", "STEREOE", "STEREOANY", "STEREONONE"])
    bond_feature_vector += stereo_type_enc
```
Encodes **E-Z isomerism** around double bonds (cis/trans configuration) - important since geometric isomers can have very different biological activity, but optional since it only applies meaningfully to double bonds.
##### Return
```python
return np.array(bond_feature_vector)
```
The flat list is cast to a **NumPy array** - a convenient per-atom (or per-bond) format that gets stacked across all atoms/bonds and converted to a `torch.Tensor` when building the graph's `Data` object.

#### Addtional steps : needed by DGCT model

**Compute ecfp**

In [7]:
def compute_ecfp(mol, n_bits=1024, radius=2):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros(n_bits, dtype=np.float32)
    for bit in fp.GetOnBits():
        arr[bit] = 1.0
    return arr

**Compute 1489 fps : concatenation of:**

    PubChem fingerprints (GetPubChemFPs)  
    MACCS keys (GetMACCSKeysFingerprint)  
    ERG fingerprints (GetErGFingerprint)  

In [8]:
def compute_fps_1489(mol):
    pubchem_fp = GetPubChemFPs(mol)           # typically 881-dim
    maccs_fp = AllChem.GetMACCSKeysFingerprint(mol)  # 167-dim
    erg_fp = AllChem.GetErGFingerprint(mol, fuzzIncrement=0.3, maxPath=21, minPath=1)  # ~441-dim
    
    fp = np.concatenate([pubchem_fp, maccs_fp, erg_fp])
    # fp should be shape (1489,)
    return fp.astype(np.float32)

#### Step 3: Generating labeled Pytorch Geometric Graph Objects

In [9]:
def create_pytorch_geometric_graph_data_list_from_smiles_and_labels(x_smiles, y):
    """
    Inputs:
    
    x_smiles = [smiles_1, smiles_2, ....] ... a list of SMILES strings
    y = [y_1, y_2, ...] ... a list of numerical labels for the SMILES strings (such as associated pKi values)
    
    Outputs:
    
    data_list = [G_1, G_2, ...] ... a list of torch_geometric.data.Data objects which represent labeled molecular graphs that can readily be used for machine learning
    
    """
    
    data_list = []
    
    for (smiles, y_val) in zip(x_smiles, y):
        
        # convert SMILES to RDKit mol object
        mol = Chem.MolFromSmiles(smiles)

        # get feature dimensions
        n_nodes = mol.GetNumAtoms()
        n_edges = 2*mol.GetNumBonds()
        unrelated_smiles = "O=O"
        unrelated_mol = Chem.MolFromSmiles(unrelated_smiles)
        n_node_features = len(get_atom_features(unrelated_mol.GetAtomWithIdx(0)))
        n_edge_features = len(get_bond_features(unrelated_mol.GetBondBetweenAtoms(0,1)))

        # construct node feature matrix X of shape (n_nodes, n_node_features)
        X = np.zeros((n_nodes, n_node_features))

        for atom in mol.GetAtoms():
            X[atom.GetIdx(), :] = get_atom_features(atom)
            
        X = torch.tensor(X, dtype = torch.float)
        
        # construct edge index array E of shape (2, n_edges)
        (rows, cols) = np.nonzero(GetAdjacencyMatrix(mol))
        torch_rows = torch.from_numpy(rows.astype(np.int64)).to(torch.long)
        torch_cols = torch.from_numpy(cols.astype(np.int64)).to(torch.long)
        E = torch.stack([torch_rows, torch_cols], dim = 0)
        
        # construct edge feature array EF of shape (n_edges, n_edge_features)
        EF = np.zeros((n_edges, n_edge_features))
        
        for (k, (i,j)) in enumerate(zip(rows, cols)):
            
            EF[k] = get_bond_features(mol.GetBondBetweenAtoms(int(i),int(j)))
        
        EF = torch.tensor(EF, dtype = torch.float)
        
        # construct label tensor
        y_tensor = torch.tensor(np.array([y_val]), dtype = torch.float)
        
        # --- NEW: compute fingerprints ---
        ecfp = compute_ecfp(mol, n_bits=1024, radius=2)       # [1024]
        fps = compute_fps_1489(mol)                            # [1489]
        w = np.array([1.0], dtype=np.float32)                  # [1]
        
        ecfp_tensor = torch.tensor(ecfp, dtype=torch.float)
        fps_tensor = torch.tensor(fps, dtype=torch.float)
        w_tensor = torch.tensor(w, dtype=torch.float)
        # ---------------------------------
        
        # construct Pytorch Geometric data object and append to data list
        data_list.append(
            Data(
                x = X,
                edge_index = E,
                edge_attr = EF,
                y = y_tensor,
                ecfp = ecfp_tensor,
                fps = fps_tensor,
                w = w_tensor,
            )
        )

    return data_list

#### SMILES-to-Graph Dataset Builder
The `create_pytorch_geometric_graph_data_list_from_smiles_and_labels` function ties everything together: it takes a list of **SMILES strings** and their **labels** (e.g. pKi values), and produces a list of **PyTorch Geometric `Data` objects** ready for GNN training.
##### Inputs / Outputs
```python
x_smiles = [smiles_1, smiles_2, ...]   # list of SMILES strings
y = [y_1, y_2, ...]                    # list of numerical labels
# returns: data_list = [G_1, G_2, ...] # list of torch_geometric.data.Data objects
```
##### 1. SMILES → RDKit mol
```python
mol = Chem.MolFromSmiles(smiles)
```
Parses the SMILES string into an **RDKit mol object**, the input format `get_atom_features` and `get_bond_features` expect.
##### 2. Determine feature dimensions dynamically
```python
n_nodes = mol.GetNumAtoms()
n_edges = 2*mol.GetNumBonds()
unrelated_smiles = "O=O"
unrelated_mol = Chem.MolFromSmiles(unrelated_smiles)
n_node_features = len(get_atom_features(unrelated_mol.GetAtomWithIdx(0)))
n_edge_features = len(get_bond_features(unrelated_mol.GetBondBetweenAtoms(0,1)))
```
`n_edges` is **doubled** because RDKit bonds are undirected but graph edges are stored **bidirectionally** (each bond → two directed edges). A throwaway molecule (`O=O`, oxygen gas) is used purely to **probe the feature vector length** without hardcoding dimensions - a robust trick if you change the feature functions later.
##### 3. Build node feature matrix `X`
```python
X = np.zeros((n_nodes, n_node_features))
for atom in mol.GetAtoms():
    X[atom.GetIdx(), :] = get_atom_features(atom)
X = torch.tensor(X, dtype = torch.float)
```
Loops over every atom, filling a **(n_nodes × n_node_features)** matrix, then converts it to a `torch.Tensor` - this becomes `Data.x`.
##### 4. Build edge index `E`
```python
(rows, cols) = np.nonzero(GetAdjacencyMatrix(mol))
torch_rows = torch.from_numpy(rows.astype(np.int64)).to(torch.long)
torch_cols = torch.from_numpy(cols.astype(np.int64)).to(torch.long)
E = torch.stack([torch_rows, torch_cols], dim = 0)
```
`GetAdjacencyMatrix` gives a binary matrix of which atoms are bonded; `np.nonzero` extracts the **(row, col) index pairs** of every connection (symmetric, so each bond appears twice - i→j and j→i). Stacked into shape **(2, n_edges)** - this becomes `Data.edge_index`, PyG's required connectivity format.
##### 5. Build edge feature matrix `EF`
```python
EF = np.zeros((n_edges, n_edge_features))
for (k, (i,j)) in enumerate(zip(rows, cols)):
    EF[k] = get_bond_features(mol.GetBondBetweenAtoms(int(i),int(j)))
EF = torch.tensor(EF, dtype = torch.float)
```
For each **(i, j)** pair from step 4, looks up the actual bond and computes its features - ensuring `EF` rows are aligned **1-to-1** with `E` columns. Becomes `Data.edge_attr`.
##### 6. Build label tensor
```python
y_tensor = torch.tensor(np.array([y_val]), dtype = torch.float)
```
Wraps the scalar label (e.g. a pKi value) into a **1-element tensor**, becomes `Data.y`.
##### 7. Assemble and collect
```python
data_list.append(Data(x = X, edge_index = E, edge_attr = EF, y = y_tensor))
```
Combines all four tensors into a single **PyG `Data` object** representing one labeled molecular graph, and appends it to the growing `data_list`.
##### Return
```python
return data_list
```
A list of `Data` objects - one per molecule - directly usable with a PyG `DataLoader` for **batched GNN training**.

**Let's build the molecular graph representation `data object` aka Geo pytorch object :**

In [10]:
"""
Build molecular graph Data objects for cdk2_compounds_df using the
get_atom_features / get_bond_features / create_pytorch_geometric_graph_data_list_from_smiles_and_labels
functions already defined in your notebook (from the blopig blog).
"""

# 1. Extract SMILES and labels from your dataframe
#    Adjust the column name if your SMILES column is called something else
#    (e.g. "canonical_smiles", "smiles", "SMILES").
x_smiles = cdk2_compounds_df["canonical_smiles"].tolist()
y = cdk2_compounds_df["pIC50"].tolist()

# 2. Call the blog's function to build the list of PyG Data objects
graphs = create_pytorch_geometric_graph_data_list_from_smiles_and_labels(x_smiles, y)

[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerat

[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerat

[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerator
[12:06:55] DEPRECATION WARNING: please use MorganGenerat

[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerat

[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerator
[12:06:56] DEPRECATION WARNING: please use MorganGenerat

...

[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerat

[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerator
[12:07:24] DEPRECATION WARNING: please use MorganGenerat

[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerat

[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerat

[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerator
[12:07:25] DEPRECATION WARNING: please use MorganGenerat

In [11]:
# Quick sanity checks
print(f"Total molecules in dataframe: {len(cdk2_compounds_df)}")
print(f"Total graphs built: {len(graphs)}")

print(f"\nFirst molecule graph: {graphs[0]}")
print("Node feature dim:", graphs[0].x.shape[1])
print("Edge feature dim:", graphs[0].edge_attr.shape[1])
print("Number of atoms in first molecule:", graphs[0].x.shape[0])
print("Number of directed edges in first molecule:", graphs[0].edge_index.shape[1])

Total molecules in dataframe: 5074
Total graphs built: 5074

First molecule graph: Data(x=[27, 79], edge_index=[2, 58], edge_attr=[58, 10], y=[1], ecfp=[1024], fps=[1489], w=[1])
Node feature dim: 79
Edge feature dim: 10
Number of atoms in first molecule: 27
Number of directed edges in first molecule: 58


In [12]:
print(graphs[0])
print("Keys in Data object:", list(graphs[0].keys()))

Data(x=[27, 79], edge_index=[2, 58], edge_attr=[58, 10], y=[1], ecfp=[1024], fps=[1489], w=[1])
Keys in Data object: ['ecfp', 'edge_index', 'x', 'fps', 'edge_attr', 'y', 'w']


### 3. Split train/val/test

#### Why Random Split Is Not Suitable - Scaffold Split Instead
A **random split** assigns molecules to train/valid/test purely by chance. In molecular datasets (like CDK2 ligand data), many compounds come from the same lead-optimization series and share the same **core scaffold**, differing only by small substituents. A random split routinely places near-identical analogs in both train and test sets, so the model can effectively **memorize the scaffold** rather than learn generalizable structure-activity relationships. This causes **inflated, unrealistic test performance** that does not hold up on genuinely novel chemotypes - which is exactly what virtual screening needs to generalize to.
##### The fix: scaffold split
A **scaffold split** groups molecules by their **Bemis-Murcko scaffold** (core ring system with side chains stripped), then assigns entire scaffold groups to train/valid/test - never splitting a scaffold group across sets. This ensures the test set contains **chemotypes unseen during training**, giving a much more realistic estimate of generalization to new chemical series.
##### Implementation used below
The next cell implements a **custom scaffold-balanced split** (`scaffold_balanced_split`) rather than DeepChem's `dc.splits.ScaffoldSplitter`. It groups molecules by `MurckoScaffold.MurckoScaffoldSmiles`, separates scaffold groups into "big" (>5 molecules) and "small" (≤5 molecules) buckets, shuffles each bucket independently (seeded for reproducibility), then fills the train/valid/test sets in group order until the 80/10/10 cutoffs are reached - keeping every scaffold group intact within a single split.
```python
from rdkit.Chem.Scaffolds import MurckoScaffold
scaffold = MurckoScaffold.MurckoScaffoldSmiles(smiles=smi, includeChirality=False)
```
Reference: Bemis, G. W., and Murcko, M. A. *"The properties of known drugs. 1. Molecular frameworks."* Journal of Medicinal Chemistry 39.15 (1996): 2887-2893.

In [13]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
import random

def scaffold_balanced_split(graphs, smiles_list, frac_train=0.8, frac_valid=0.1, seed=42):
    scaffolds = defaultdict(list)
    for i, smi in enumerate(smiles_list):
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(smiles=smi, includeChirality=False)
        scaffolds[scaffold].append(i)

    rng = random.Random(seed)
    big_groups = [g for g in scaffolds.values() if len(g) > 5]
    small_groups = [g for g in scaffolds.values() if len(g) <= 5]
    rng.shuffle(big_groups)
    rng.shuffle(small_groups)
    ordered_groups = big_groups + small_groups

    n_total = len(smiles_list)
    train_cutoff = frac_train * n_total
    valid_cutoff = (frac_train + frac_valid) * n_total

    train_idx, valid_idx, test_idx = [], [], []
    count = 0
    for group in ordered_groups:
        count += len(group)
        if count <= train_cutoff:
            train_idx.extend(group)
        elif count <= valid_cutoff:
            valid_idx.extend(group)
        else:
            test_idx.extend(group)
    return train_idx, valid_idx, test_idx

# Call it using your existing variables
train_idx, valid_idx, test_idx = scaffold_balanced_split(
    graphs,
    cdk2_compounds_df["canonical_smiles"].tolist(),
    frac_train=0.8,
    frac_valid=0.1,
    seed=42
)

# Use the returned indices to pull the matching PyTorch Geometric Data
# objects out of our original `graphs` list - this is the step that finally
# turns "which rows go where" into three usable graph datasets for training,
# validation, and testing.
train_graphs = [graphs[i] for i in train_idx]
valid_graphs = [graphs[i] for i in valid_idx]
test_graphs  = [graphs[i] for i in test_idx]

# Sanity check: confirm the split sizes roughly match the requested
# 80/10/10 fractions, and that no molecules were silently dropped
# (len(train)+len(valid)+len(test) should equal len(graphs)).
print(f"Train: {len(train_graphs)}, Valid: {len(valid_graphs)}, Test: {len(test_graphs)}")

Train: 4059, Valid: 507, Test: 508


### 4.GNN architecture

Two independent GNN pipelines are trained and evaluated in this section on the same scaffold split, so their test metrics are directly comparable:

- **4.1 AttentionDGCT**  dual-encoder GAT + GIN fusion model (`Model_gnn_fp`) combined with PubChem/MACCS/ERG fingerprints.
- **4.2 AttentiveFP**  single attention-based GNN operating on atom/bond graph features only, with GRU-updated atom and molecule embeddings.

#### 4.1 AttentionDGCT (AttentionDGCL: GAT + GIN fusion with fingerprints)

**Models used in the cells below**

This section builds and trains a dual-branch graph neural network that fuses two complementary graph encoders with precomputed molecular fingerprints (the model referred to as `Model_gnn_fp`, labeled `AttentionDGCL` in the code comments).

- **GATNet** (Graph Attention Network) - two stacked `GATConv` layers (79 → 510 → 510 channels, 10 attention heads) followed by a linear projection to a 512-dim graph embedding. Attention weights let it focus on the most relevant neighboring atoms.
- **GINNet** (Graph Isomorphism Network with edge features) - two `GINEConv` layers, each wrapping a small MLP (Linear → ReLU → Linear), that incorporate bond (edge) features directly and also output a 512-dim graph embedding. GIN is as expressive as the Weisfeiler-Lehman test, which helps it distinguish subtly different molecular topologies.
- **Model_gnn_fp** - the downstream fusion model. It concatenates the GATNet and GINNet graph embeddings (1024-dim) with a dense branch that processes the 1489-dim fingerprint vector (PubChem + MACCS + ERG), then feeds the combined 1536-dim representation through a fully connected regression head (Linear/BatchNorm/ReLU/Dropout) to predict a single pIC50 value.

The next cells inspect each class's constructor/forward signatures, instantiate the two encoders and the fusion model, build the DataLoaders, and run the training loop (Adam optimizer, MSE loss, early stopping on validation MSE).

##### 4.1.1 Inspect encoder & model signatures

In [14]:
import inspect
from encoder_gnn import GATNet, GINNet

print("GATNet signature:")
print(inspect.signature(GATNet.__init__))

print("\nGINNet signature:")
print(inspect.signature(GINNet.__init__))

GATNet signature:
(self, num_features_xd=79, output_dim=512, heads=10, edge_dim=10, dropout=0.2)

GINNet signature:
(self, num_features_xd=79, output_dim=512, edge_dim=10, eps=0.0, train_eps=True, dropout=0.2)


In [18]:
import inspect
from model_gnn_fp_downstream import Model_gnn_fp

print("Model_gnn_fp signature:")
print(inspect.signature(Model_gnn_fp.__init__))

Model_gnn_fp signature:
(self, n_output=1, output_dim=512, dropout=0.2, encoder1=None, encoder2=None, fp_dim=1489)


In [19]:
import inspect
from model_gnn_fp_downstream import Model_gnn_fp

print("Model_gnn_fp.forward signature:")
print(inspect.signature(Model_gnn_fp.forward))

Model_gnn_fp.forward signature:
(self, data)


##### 4.1.2 Set device

In [20]:
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from sklearn.metrics import r2_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


##### 4.1.3 Instantiate encoders and fusion model

In [21]:
from encoder_gnn import GATNet, GINNet
from model_gnn_fp_downstream import Model_gnn_fp

In [22]:
# Encoders
model_encoder1 = GATNet(
    num_features_xd=79,
    output_dim=512,
    heads=10,
    edge_dim=10,
    dropout=0.2
).to(device)

model_encoder2 = GINNet(
    num_features_xd=79,
    output_dim=512,
    edge_dim=10,
    eps=0.0,
    train_eps=True,
    dropout=0.2
).to(device)

# Full AttentionDGCL model
model = Model_gnn_fp(
    n_output=1,
    output_dim=512,
    dropout=0.2,
    encoder1=model_encoder1,
    encoder2=model_encoder2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

print("Model ready.")
print(model)

Model ready.
Model_gnn_fp(
  (encoder1): GATNet(
    (gat1): GATConv(79, 51, heads=10)
    (gat2): GATConv(510, 51, heads=10)
    (fc): Linear(in_features=510, out_features=512, bias=True)
  )
  (encoder2): GINNet(
    (gine1): GINEConv(nn=Sequential(
      (0): Linear(in_features=79, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=512, bias=True)
    ))
    (gine2): GINEConv(nn=Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=512, bias=True)
    ))
    (fc): Linear(in_features=512, out_features=512, bias=True)
  )
  (fc): Sequential(
    (0): Linear(in_features=1489, out_features=1024, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=1024, out_features=512, bias=True)
  )
  (weight_fc): Linear(in_featur

##### 4.1.4 Build data loaders

In [23]:
batch_size = 32

train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=batch_size)
test_loader  = DataLoader(test_graphs,  batch_size=batch_size)

print("Train:", len(train_loader.dataset),
      "Valid:", len(valid_loader.dataset),
      "Test:", len(test_loader.dataset))

Train: 4059 Valid: 507 Test: 508


##### 4.1.5 Training loop (early stopping) + test evaluation

In [24]:
epochs = 200
best_val_mse = float('inf')
patience = 20
trigger_times = 0

for epoch in range(1, epochs+1):
    # Train
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # New forward signature
        out, y, w, out_loss, y_loss, w_loss = model(batch)

        loss = loss_fn(out_loss, y_loss)

        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(y_loss)

    train_loss /= len(train_loader.dataset)

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in valid_loader:
            batch = batch.to(device)

            out, y, w, out_loss, y_loss, w_loss = model(batch)

            loss = loss_fn(out_loss, y_loss)
            val_loss += loss.item() * len(y_loss)

    val_loss /= len(valid_loader.dataset)
    val_mse = val_loss

    print(f"Epoch {epoch:03d} | Train MSE: {train_loss:.5f} | Val MSE: {val_mse:.5f}")

    # Early stopping + save best
    if val_mse < best_val_mse:
        best_val_mse = val_mse
        trigger_times = 0
        torch.save(model.state_dict(), "attention_dgcl_cdk2_best.pth")
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break

# Load best model and evaluate on test
model.load_state_dict(torch.load("attention_dgcl_cdk2_best.pth", weights_only=True))
model.eval()

all_y_true = []
all_y_pred = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)

        out, y, w, out_loss, y_loss, w_loss = model(batch)

        all_y_true.extend(y_loss.cpu().numpy().tolist())
        all_y_pred.extend(out_loss.cpu().numpy().tolist())

y_true = np.array(all_y_true)
y_pred = np.array(all_y_pred)

rmse = np.sqrt(np.mean((y_true - y_pred)**2))
mae = np.mean(np.abs(y_true - y_pred))
r2 = r2_score(y_true, y_pred)

print(f"Test -> RMSE: {rmse:.5f}, R2: {r2:.5f}, MAE: {mae:.5f}")

Epoch 001 | Train MSE: 6.29104 | Val MSE: 1.37359


Epoch 002 | Train MSE: 1.44863 | Val MSE: 1.03873


Epoch 003 | Train MSE: 1.39434 | Val MSE: 1.34993


Epoch 004 | Train MSE: 1.22538 | Val MSE: 1.14638


Epoch 005 | Train MSE: 1.15281 | Val MSE: 1.14521


...

Epoch 048 | Train MSE: 0.66088 | Val MSE: 1.01941


Epoch 049 | Train MSE: 0.62987 | Val MSE: 0.95230


Epoch 050 | Train MSE: 0.63598 | Val MSE: 1.42651


Epoch 051 | Train MSE: 0.62915 | Val MSE: 1.17031


Epoch 052 | Train MSE: 0.65121 | Val MSE: 1.12759
Early stopping triggered.
Test -> RMSE: 1.06854, R2: 0.22485, MAE: 0.79024


**Interpretation  AttentionDGCL (GAT+GIN+fingerprint fusion) results**

- Training MSE drops smoothly and monotonically (6.29 → ~0.63 over 52 epochs), but validation MSE never settles  it oscillates between roughly 0.89 and 1.47 with no clear downward trend after epoch ~20. This is the classic small-dataset overfitting signature: the model (three fused 512-dim branches feeding a 5-layer MLP head) has more capacity than the ~4059 training molecules can reliably constrain.
- Early stopping (patience 20) correctly caught this and rolled back to the epoch-32 checkpoint (best val MSE ≈ 0.897).
- **Test set: RMSE 1.06854, R² 0.22485, MAE 0.79024**  the model explains only ~22% of the variance in held-out pIC50 values. An MAE of ~0.79 log units corresponds to roughly a 6-fold average error in predicted IC50 potency  usable for coarse virtual-screening triage/ranking, but not for fine-grained potency prediction.
- For reference, the original (pre-file-loss) run of this same architecture scored **Test RMSE 1.008, R² 0.310, MAE 0.763**  noticeably better than this reconstructed version. The ~0.08 R² gap is larger than typical run-to-run seed noise, suggesting the reconstructed `encoder_gnn.py`/`model_gnn_fp_downstream.py` are structurally similar but not functionally identical to the original (likely differences in exact attention-fusion mechanics or dropout/layer placement).

### 4.2 AttentiveFP (with edge features)

paper : https://doi.org/10.1039/d5dd00407a  
github : https://github.com/Yangxin666/HASolGNN

**Model used in the cell below**

This section implements a second, independent GNN architecture - **AttentiveFP** - as an alternative to the GAT/GIN fusion model from section 4. Unlike `Model_gnn_fp`, it works directly on the raw atom/bond feature graph (no fingerprint branch) and uses a two-stage attention mechanism to build both atom-level and molecule-level embeddings:

- **GATEConv** - a custom `MessagePassing` layer used only for the first atom-embedding step. It gates each neighbor's message by concatenating its features with the edge (bond) features, then weights neighbors with a learned attention coefficient (`softmax` over incoming edges), similar in spirit to `GATConv` but edge-aware from the start.
- **AttentiveFP** - the full model class. It first projects the 79-dim atom features into a hidden space (`Linear`), refines them with the `GATEConv` layer plus a `GRUCell` update, then stacks `num_layers - 1` additional rounds of plain `GATConv` + `GRUCell` to propagate information further across the molecular graph (atom embedding). It then performs `num_timesteps` rounds of attention-based pooling from atoms to a single virtual "super node" representing the whole molecule, again updated with a `GRUCell` (molecule embedding). A final `Linear` layer maps the molecule embedding to the single pIC50 output.
- This design follows Xiong et al.'s *"Pushing the Boundaries of Molecular Representation for Drug Discovery with the Graph Attention Mechanism"* (the architecture paper linked above), and is separately re-implemented here (`GATEConv`/`AttentiveFP` classes) rather than imported from `encoder_gnn.py`.

The next cells (4.2–4.4) instantiate this `AttentiveFP` model, set up its optimizer, build fresh DataLoaders, and define a `train_epochs` training loop with early stopping - run independently of the GAT/GIN fusion model trained in section 4.

In [32]:
from typing import Optional

import torch
import torch.nn.functional as F
from torch import Tensor
from torch.nn import GRUCell, Linear, Parameter

from torch_geometric.nn import GATConv, GATv2Conv, MessagePassing, global_add_pool, global_mean_pool
from torch_geometric.nn.inits import glorot, zeros
from torch_geometric.typing import Adj, OptTensor
from torch_geometric.utils import softmax


class GATEConv(MessagePassing):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        edge_dim: int,
        dropout: float = 0.0,
    ):
        super().__init__(aggr='add', node_dim=0)

        self.dropout = dropout

        self.att_l = Parameter(torch.empty(1, out_channels))
        self.att_r = Parameter(torch.empty(1, in_channels))

        self.lin1 = Linear(in_channels + edge_dim, out_channels, False)
        self.lin2 = Linear(out_channels, out_channels, False)

        self.bias = Parameter(torch.empty(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        glorot(self.att_l)
        glorot(self.att_r)
        glorot(self.lin1.weight)
        glorot(self.lin2.weight)
        zeros(self.bias)

    def forward(self, x: Tensor, edge_index: Adj, edge_attr: Tensor) -> Tensor:
        # edge_updater_type: (x: Tensor, edge_attr: Tensor)
        alpha = self.edge_updater(edge_index, x=x, edge_attr=edge_attr)

        # propagate_type: (x: Tensor, alpha: Tensor)
        out = self.propagate(edge_index, x=x, alpha=alpha)
        out = out + self.bias
        return out

    def edge_update(self, x_j: Tensor, x_i: Tensor, edge_attr: Tensor,
                    index: Tensor, ptr: OptTensor,
                    size_i: Optional[int]) -> Tensor:
        x_j = F.leaky_relu_(self.lin1(torch.cat([x_j, edge_attr], dim=-1)))
        alpha_j = (x_j @ self.att_l.t()).squeeze(-1)
        alpha_i = (x_i @ self.att_r.t()).squeeze(-1)
        alpha = alpha_j + alpha_i
        alpha = F.leaky_relu_(alpha)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        return alpha

    def message(self, x_j: Tensor, alpha: Tensor) -> Tensor:
        return self.lin2(x_j) * alpha.unsqueeze(-1)


class AttentiveFP(torch.nn.Module):
    r"""The Attentive FP model for molecular representation learning from the
    `"Pushing the Boundaries of Molecular Representation for Drug Discovery
    with the Graph Attention Mechanism"
    <https://pubs.acs.org/doi/10.1021/acs.jmedchem.9b00959>`_ paper, based on
    graph attention mechanisms.

    Args:
        in_channels (int): Size of each input sample.
        hidden_channels (int): Hidden node feature dimensionality.
        out_channels (int): Size of each output sample.
        edge_dim (int): Edge feature dimensionality.
        num_layers (int): Number of GNN layers.
        num_timesteps (int): Number of iterative refinement steps for global
            readout.
        dropout (float, optional): Dropout probability. (default: :obj:`0.0`)

    """
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        out_channels: int,
        edge_dim: int,
        num_layers: int,
        num_timesteps: int,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.num_layers = num_layers
        self.num_timesteps = num_timesteps
        self.dropout = dropout

        self.lin1 = Linear(in_channels, hidden_channels)

        self.gate_conv = GATEConv(hidden_channels, hidden_channels, edge_dim,
                                  dropout)
        self.gru = GRUCell(hidden_channels, hidden_channels)

        self.atom_convs = torch.nn.ModuleList()
        self.atom_grus = torch.nn.ModuleList()
        for _ in range(num_layers - 1):
            conv = GATConv(hidden_channels, hidden_channels, dropout=dropout,
                           add_self_loops=False, negative_slope=0.01)
            self.atom_convs.append(conv)
            self.atom_grus.append(GRUCell(hidden_channels, hidden_channels))

        self.mol_conv = GATConv(hidden_channels, hidden_channels,
                                dropout=dropout, add_self_loops=False,
                                negative_slope=0.01)
        self.mol_conv.explain = False  # Cannot explain global pooling.
        self.mol_gru = GRUCell(hidden_channels, hidden_channels)

        self.lin2 = Linear(hidden_channels, out_channels)

        self.reset_parameters()

    def reset_parameters(self):
        r"""Resets all learnable parameters of the module."""
        self.lin1.reset_parameters()
        self.gate_conv.reset_parameters()
        self.gru.reset_parameters()
        for conv, gru in zip(self.atom_convs, self.atom_grus):
            conv.reset_parameters()
            gru.reset_parameters()
        self.mol_conv.reset_parameters()
        self.mol_gru.reset_parameters()
        self.lin2.reset_parameters()

    def forward(self, x: Tensor, edge_index: Tensor, edge_attr: Tensor,
                batch: Tensor) -> Tensor:
        """"""  # noqa: D419
        # Atom Embedding:
        x = F.leaky_relu_(self.lin1(x))

        h = F.elu_(self.gate_conv(x, edge_index, edge_attr))
        h = F.dropout(h, p=self.dropout, training=self.training)
        x = self.gru(h, x).relu_()

        for conv, gru in zip(self.atom_convs, self.atom_grus):
            h = conv(x, edge_index)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = gru(h, x).relu()

        # Molecule Embedding:
        row = torch.arange(batch.size(0), device=batch.device)
        edge_index = torch.stack([row, batch], dim=0)

        out = global_add_pool(x, batch).relu_()
        for t in range(self.num_timesteps):
            h = F.elu_(self.mol_conv((x, out), edge_index))
            h = F.dropout(h, p=self.dropout, training=self.training)
            out = self.mol_gru(h, out).relu_()

        # Predictor:
        #concatenate graph-level features to the output layer
        # out = torch.concat((out, graph_level_features.unsqueeze(0)), dim = -1)
        # out = F.dropout(out, p=self.dropout, training=self.training)
        return self.lin2(out)

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}('
                f'in_channels={self.in_channels}, '
                f'hidden_channels={self.hidden_channels}, '
                f'out_channels={self.out_channels}, '
                f'edge_dim={self.edge_dim}, '
                f'num_layers={self.num_layers}, '
                f'num_timesteps={self.num_timesteps}'
                f')')

##### 4.2.1 Instantiate the model

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AttentiveFP(
    in_channels=79,     # matches your atom_features() output width
    hidden_channels=64, # your choice - controls model capacity
    out_channels=1,     # single scalar output: pIC50
    edge_dim=10,          # matches your bond_features() output width
    num_layers=3,
    num_timesteps=3,
    dropout=0.2,
).to(device)

##### 4.2.2 Instantiate the optimizer

In [34]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

##### 4.2.3 Data loaders, loss function, and training loop

In [35]:
train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=32)
test_loader = DataLoader(test_graphs, batch_size=32)

In [36]:
import math
import numpy
import torch
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr


def training(loader, model, loss_fn, optimizer, device):
    """Run one training epoch over `loader` and return (avg_loss, model)."""
    model.train()
    total_loss = torch.zeros(1, device=device)
    n_graphs = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr, data.batch).view(-1)
        loss = loss_fn(out, data.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.detach() * data.num_graphs
        n_graphs += data.num_graphs
    return (total_loss / n_graphs).squeeze(), model


def validation(loader, model, loss_fn, device):
    """Run one evaluation epoch over `loader` and return avg_loss (no grad)."""
    model.eval()
    total_loss = torch.zeros(1, device=device)
    n_graphs = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch).view(-1)
            loss = loss_fn(out, data.y.view(-1))
            total_loss += loss.detach() * data.num_graphs
            n_graphs += data.num_graphs
    return (total_loss / n_graphs).squeeze()


def train_epochs(epochs, model, train_loader, val_loader, path, device):
    """Training over all epochs

    Args:
        epochs (int): number of epochs to train for
        model (nn.Module): the current model
        train_loader (DataLoader): training data in batches
        val_loader (DataLoader): validation data in batches
        path (string): path to save the best model
        device (torch.device): device to run training on

    Returns:
        dict: train/val losses over all epochs, plus MAE, RMSE, and Spearman
              correlation for the last epoch's train predictions and for the
              final validation predictions, along with raw prediction/target
              arrays for both splits.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    loss = torch.nn.MSELoss()

    train_target = numpy.empty((0))
    train_y_target = numpy.empty((0))
    train_loss = numpy.empty(epochs)
    val_loss = numpy.empty(epochs)
    best_loss = math.inf

    for epoch in range(epochs):
        epoch_loss, model = training(train_loader, model, loss, optimizer, device)
        v_loss = validation(val_loader, model, loss, device)
        if v_loss < best_loss:
            best_loss = v_loss
            torch.save(model.state_dict(), path)

        if epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                for d in train_loader:
                    d = d.to(device)
                    out = model(d.x, d.edge_index, d.edge_attr, d.batch).view(-1)
                    train_target = numpy.concatenate((train_target, out.cpu().numpy()))
                    train_y_target = numpy.concatenate((train_y_target, d.y.cpu().numpy()))
            model.train()

        train_loss[epoch] = epoch_loss.detach().cpu().numpy()
        val_loss[epoch] = v_loss.detach().cpu().numpy()

        if epoch % 2 == 0:
            print(
                "Epoch: " + str(epoch)
                + ", Train loss: " + str(epoch_loss.item())
                + ", Val loss: " + str(v_loss.item())
            )

    # --- Collect validation predictions/targets from the final model state ---
    val_target = numpy.empty((0))
    val_y_target = numpy.empty((0))
    model.eval()
    with torch.no_grad():
        for d in val_loader:
            d = d.to(device)
            out = model(d.x, d.edge_index, d.edge_attr, d.batch).view(-1)
            val_target = numpy.concatenate((val_target, out.cpu().numpy()))
            val_y_target = numpy.concatenate((val_y_target, d.y.cpu().numpy()))
    model.train()

    # --- Metrics on last-epoch train predictions ---
    train_mae = mean_absolute_error(train_y_target, train_target)
    train_rmse = numpy.sqrt(numpy.mean((train_y_target - train_target) ** 2))
    train_r2 = r2_score(train_y_target, train_target)
    train_spearman = spearmanr(train_y_target, train_target).correlation

    # --- Metrics on validation predictions ---
    val_mae = mean_absolute_error(val_y_target, val_target)
    val_rmse = numpy.sqrt(numpy.mean((val_y_target - val_target) ** 2))
    val_r2 = r2_score(val_y_target, val_target)
    val_spearman = spearmanr(val_y_target, val_target).correlation

    print("\nFinal Train Metrics -> MAE: {:.5f}, RMSE: {:.5f}, R2: {:.5f}, Spearman: {:.5f}".format(
        train_mae, train_rmse, train_r2, train_spearman))
    print("Final Val Metrics   -> MAE: {:.5f}, RMSE: {:.5f}, R2: {:.5f}, Spearman: {:.5f}".format(
        val_mae, val_rmse, val_r2, val_spearman))

    return {
        "train_loss": train_loss, "val_loss": val_loss,
        "train_target": train_target, "train_y_target": train_y_target,
        "val_target": val_target, "val_y_target": val_y_target,
        "train_mae": train_mae, "train_rmse": train_rmse,
        "train_r2": train_r2, "train_spearman": train_spearman,
        "val_mae": val_mae, "val_rmse": val_rmse,
        "val_r2": val_r2, "val_spearman": val_spearman,
    }

##### 4.2.4 Run training

In [30]:
results = train_epochs(
    epochs=200,
    model=model,
    train_loader=train_loader,
    val_loader=valid_loader,
    path="attentivefp_cdk2_best.pth",
    device=device,
)

Epoch: 0, Train loss: 4.734846591949463, Val loss: 1.5904546976089478


Epoch: 2, Train loss: 1.432257890701294, Val loss: 1.7297922372817993


Epoch: 4, Train loss: 1.2542645931243896, Val loss: 1.3083875179290771


Epoch: 6, Train loss: 1.1867252588272095, Val loss: 1.3768999576568604


Epoch: 8, Train loss: 1.114880084991455, Val loss: 1.2573810815811157


...

Epoch: 192, Train loss: 0.5451450347900391, Val loss: 0.881589949131012


Epoch: 194, Train loss: 0.564638614654541, Val loss: 0.8779565095901489


Epoch: 196, Train loss: 0.5436056852340698, Val loss: 0.8863004446029663


Epoch: 198, Train loss: 0.5567026138305664, Val loss: 0.9211593866348267



Final Train Metrics -> MAE: 0.53093, RMSE: 0.69615, R2: 0.68317, Spearman: 0.77046
Final Val Metrics   -> MAE: 0.72144, RMSE: 0.95755, R2: 0.41937, Spearman: 0.58698


##### 4.2.5 Evaluate on the held-out test set

In [31]:
model.load_state_dict(torch.load("attentivefp_cdk2_best.pth", weights_only=True))
model.eval()

all_y_true, all_y_pred = [], []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.edge_attr, data.batch).view(-1)
        all_y_true.extend(data.y.cpu().numpy().tolist())
        all_y_pred.extend(out.cpu().numpy().tolist())

y_true = np.array(all_y_true)
y_pred = np.array(all_y_pred)

rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
mae = np.mean(np.abs(y_true - y_pred))
r2 = r2_score(y_true, y_pred)

print(f"AttentiveFP Test -> RMSE: {rmse:.5f}, R2: {r2:.5f}, MAE: {mae:.5f}")

AttentiveFP Test -> RMSE: 1.06290, R2: 0.23301, MAE: 0.80220


**Interpretation  AttentiveFP results & comparison to AttentionDGCL**

- AttentiveFP trained more smoothly than the fusion model (train loss decreases almost monotonically across all 200 epochs, never triggering early stopping), but the **train → val → test R² cascade (0.683 → 0.419 → 0.233)** is a textbook overfitting gradient: the model fits training chemotypes well, generalizes moderately to validation scaffolds, and drops further on the completely held-out test scaffolds.
- **Test set: RMSE 1.06290, R² 0.23301, MAE 0.80220.**

***Both architectures converge to nearly identical test performance:**

| Metric | AttentionDGCL (4.1) | AttentiveFP (4.2) |
|---|---|---|
| Test RMSE | 1.069 | 1.063 |
| Test R² | 0.225 | 0.233 |
| Test MAE | 0.790 | 0.802 |

Despite very different architectures  one fuses two graph encoders with 1489-dim PubChem/MACCS/ERG fingerprints, the other is a single attention-based GNN using only atom/bond features  their test-set performance is statistically indistinguishable. This strongly suggests the ~0.22–0.23 R² ceiling here is set by the **scaffold split** itself (deliberately testing on chemotypes structurally unlike anything in training) rather than by model capacity or architecture choice  adding fingerprints and a second encoder branch bought essentially nothing over the simpler single-model AttentiveFP.

**Data Scarcity as the Likely Root Cause**  

Both models show large train→val→test performance gaps despite very different designs, which points to data scarcity rather than architecture as the dominant limiting factor:

4,059 training molecules is small relative to the model capacity being fit (three 512-dim branches, hundreds of thousands of parameters). Published GNN benchmarks for molecular property prediction typically use datasets an order of magnitude larger, and low-data drug discovery literature consistently notes that typical assay datasets (often just hundreds to low thousands of molecules) are "drastically smaller compared to other deep learning applications," with limited structural diversity and experimental noise compounding the problem.

The scaffold split compounds this  it doesn't just shrink the effective training set, it deliberately removes entire structural families from training and places them in test, which is exactly the setup known to punish deep GNNs hardest, since they are data-hungry and struggle most on genuinely novel chemotypes.

A concrete clue: the original DGCL-main repo includes a 22 MB pretraining dataset, that isn't used in this notebook. This suggests the original AttentionDGCL pipeline was likely designed to pretrain the GAT/GIN encoders on this larger corpus first (e.g., via self-supervised masked-atom/bond prediction or contrastive learning) and then fine-tune on the small CDK2 set  not train from scratch as this notebook currently does. If the original R²=0.310 run used pretrained weights, that alone would explain most of the gap versus the from-scratch reconstructed version.